# Module 06 — Dictionaries and sets

Module 05 finished on a refusal: a tuple can be a dict key, a list cannot. This
module says why, and then spends its time on what that refusal buys.

## 1. A dict is a literal

In Java a small map is `Map.of(...)` or three `put` calls. In Python it is the
syntax itself, and it is the type you reach for first.

In [ ]:
reading = {"tag": "TH-04", "value": 91.0, "unit": "C"}

print(reading["tag"])
print(len(reading))
print("value" in reading)  # `in` on a dict asks about KEYS, not values
print(91.0 in reading)

There is no `containsKey`: `in` is it. That it looks at keys rather than values is
worth fixing in your head now, because the line reads perfectly well either way.

Asking for a key that is not there raises. Asking with `.get` does not.

In [ ]:
reading = {"tag": "TH-04", "value": 91.0}

print(reading.get("unit"))  # None -- no complaint
print(reading.get("unit", "C"))  # a default you choose

try:
    reading["unit"]
except KeyError as err:
    print("KeyError:", err)  # the message is the key itself, in quotes

`.get` is right when a missing key is an expected case. `[...]` is right when it is
not — the exception then stops the program at the line that made the wrong
assumption, instead of letting a `None` travel three functions before it fails.
Choosing `.get` everywhere is a way of turning loud bugs into quiet ones.

## 2. What can be a key

A dict finds a key by its hash and would not reliably find it again if that number
moved.
So a key must be **hashable**: it must have a hash, and that hash must not change
while the key is in use. The mutable built-in containers — `list`, `dict`, `set`,
`bytearray` — have no hash at all, which turns a future silent failure into an
immediate `TypeError`.

In [ ]:
print(hash("TH-04"))
print(hash(("TH-04", 21.7)))  # a tuple of hashables is hashable

for candidate in ([1, 2], (1, [2]), {"a": 1}):
    try:
        hash(candidate)
    except TypeError as err:
        print(f"{str(candidate):12} -> TypeError: {err}")

`(1, [2])` is the one to remember: it is a tuple, and it is still refused. A tuple
hashes by hashing its contents, so it is hashable exactly when everything in it is.
"Immutable" is not quite the rule; "hashable" is.

The refusal is worth seeing as a favour. Your own classes are hashable by default,
by identity, so mutating one that is already a key changes nothing and the lookup
still works. Write a `__hash__` that reads a field, mutate that field, and the key is
gone: the entry is still in the dict, `len` still counts it, and `key in d` is now
`False`. That is the silent failure `TypeError` is there to prevent, and it is why
the rule is about the hash rather than about mutation as such.

`frozenset` is the set that gives up changing in exchange for being usable as a key.

In [ ]:
by_pair = {frozenset({"TH-04", "TH-09"}): "shared cable"}
print(by_pair[frozenset({"TH-09", "TH-04"})])  # order does not matter to a frozenset

Keys are unique — but "unique" is decided by `==`, not by type or by how the value
was written. Two keys that compare equal and hash equal are one key.

`ruff` objects to the line below — `F601`, a repeated key literal — and it is right:
written out like this, the collision is a typo in every program but this one. The
`# noqa` overrules the rule for that line, the way module 04 did for `F811`.

Predict what the dict ends up as. Both parts of the answer matter: how many entries,
and which value survived.

In [ ]:
collapsed = {1: "int", 1.0: "float", True: "bool"}  # noqa: F601 -- the collision is the point

assert collapsed == ...

`1 == 1.0 == True` and their hashes agree, so all three are the same key. The first
one written wins the key — the dict keeps the `1` it already had rather than
replacing it — and the last one written wins the value.

This is not a curiosity you can dismiss. `True` as a key and `1` as a key are the
same key, and any dict that mixes `int` and `float` keys is capable of it.

## 3. Iterating gives you keys

A plain loop over a dict yields keys, not entries: `for tag in readings` is Java's
`for (String tag : map.keySet())`. The `entrySet()` form is `.items()`, below —
Python simply made the key form the short one.

In [ ]:
readings = {"TH-01": 21.7, "TH-04": 91.0, "TH-09": 23.1}

for tag in readings:
    print(tag, readings[tag])

print()

# .items() yields pairs, and the loop header takes them apart -- the tuple
# unpacking of module 05, in the place you will use it most.
for tag, value in readings.items():
    print(f"{tag}: {value}")

`.keys()`, `.values()` and `.items()` return **views**: they look at the dict rather
than copying it, so they follow along when it changes.

In [ ]:
readings = {"TH-01": 21.7}
tags = readings.keys()

readings["TH-04"] = 91.0

print(tags)  # the view has the new key
print(list(tags))  # list() is how you take a snapshot

Changing which keys a dict has while looping over it is refused. Java's `keySet()` is
a view too and its iterators are fail-fast, so the shape of the answer is the same in
both languages — but the checks are not the same, and neither is as solid as it
looks. The cell below is three runs of the same idea.

In [ ]:
def swap_during_loop(readings):
    """Remove the first key and add another -- the size ends up unchanged."""
    walked = []
    try:
        for position, tag in enumerate(readings):
            walked.append(tag)
            if position == 0:
                readings.pop(tag)
                readings["TH-99"] = 0.0
    except RuntimeError as err:
        return f"RuntimeError: {err}"
    return f"no error, and the loop walked {walked}"


three = {"TH-01": 21.7, "TH-04": 91.0, "TH-09": 23.1}
five = {"TH-01": 21.7, "TH-04": 91.0, "TH-09": 23.1, "TH-02": 88.4, "TH-07": 22.8}

try:
    for tag in three:
        three["TH-11"] = 1.0  # one more entry than when the loop started
except RuntimeError as err:
    print("adding:   ", err)

print("swap, 3:  ", swap_during_loop({"TH-01": 21.7, "TH-04": 91.0, "TH-09": 23.1}))
print("swap, 5:  ", swap_during_loop(five))

Read the last two lines again: **the same code, and only the size of the dict is
different.** Removing one key and adding another keeps the count, so the size check
cannot see it — and what happens next is not decided by any rule you can learn. On
the three-entry dict a second check catches it and names keys instead of size. On the
five-entry one nothing complains at all, the loop runs to the end, and one key it was
supposed to visit is simply not in the list it walked.

So the exception is a smoke alarm, not a guarantee. Java says the same thing about
itself in writing: the javadoc calls fail-fast behaviour best-effort and warns
against writing a program that depends on it. Both languages will usually catch you,
neither promises to, and the conclusion in both is the same — do not change which
keys exist while looping over them.

When the dict has to change, loop over a snapshot:

In [ ]:
readings = {"TH-01": 21.7, "TH-04": 91.0}

for tag in list(readings):  # list() takes the snapshot; the dict is then free to change
    if readings[tag] > 85:
        del readings[tag]

print(readings)

# Replacing the value of a key that is already there needs no snapshot: the set of
# keys is what it was when the loop started.
readings = {"TH-01": 21.7, "TH-04": 91.0}
for tag in readings:
    readings[tag] = round(readings[tag])

print(readings)

## 4. Insertion order is kept

Since Python 3.7 a dict keeps the order in which keys were first inserted, and that
is a guarantee of the language, not an accident of the implementation.

It says less than it seems to. It is not sorting, and re-inserting a key that was
deleted puts it at the end.

In [ ]:
order = {"b": 1, "a": 2, "c": 3}
print(order)
print(list(order))

del order["b"]
order["b"] = 9  # same key, new position
print(order)

print(sorted(order))  # sorted() on a dict gives its keys -- sorted, not in insertion order

## 5. Counting and grouping

`d[key] += 1` reads the key before it writes it, so on a key that is not there yet
it raises. Predict which error.

In [ ]:
counts = {}

try:
    counts["TH-04"] += 1
    outcome = "worked"
except Exception as err:
    outcome = type(err).__name__

assert outcome == ...

The two language-level answers, both worth being able to read:

In [ ]:
log = [("TH-04", 91.0), ("TH-01", 21.7), ("TH-04", 88.0)]

counts = {}
for tag, _ in log:
    counts[tag] = counts.get(tag, 0) + 1  # get supplies the starting value
print(counts)

grouped = {}
for tag, value in log:
    grouped.setdefault(tag, []).append(value)  # setdefault returns the list either way
print(grouped)

`setdefault` inserts the default when the key is missing and returns the value that
is now there — so the `.append` lands in the dict, whether the list was just created
or already existed. Its name is misleading and its behaviour is worth reading twice.

`collections.Counter` and `collections.defaultdict` do both of these in one line and
are in module 10. What is here is the version that needs no import, which is what
you will find in other people's code.

A dict comprehension builds one the way a list comprehension builds a list — and
inherits the collapse rule from section 2:

In [ ]:
log = [("TH-04", 91.0), ("TH-01", 21.7), ("TH-04", 88.0)]

print({tag: value for tag, value in log})  # TH-04 appears twice; the last one wins
print({value: tag for tag, value in log})  # inverted -- and now the keys are floats

## 6. Where Java writes a class

A record with three fields is a dict literal, and for reading a config file, parsing
JSON (module 09), or carrying a row from a database (module 19), that is the right
answer.

In [ ]:
sensor = {
    "tag": "TH-04",
    "unit": "C",
    "limits": {"low": -20.0, "high": 85.0},  # nesting is just a dict as a value
}

print(sensor["limits"]["high"])
print(sensor.get("calibrated", "unknown"))

Where this stops being the right answer: the field names are now strings, so nothing
checks them. `sensor["untit"]` is a `KeyError` at runtime where a class would have
been a red squiggle in the editor, a plain dict gives `mypy` nothing to check the
keys against, and the shape of the record lives in whoever last wrote to it.
(`typing.TypedDict` buys the checking back without giving up the dict —
`solutions/solution_07.md` has it.)

The rule of thumb: a dict when the keys are **data** — read from somewhere, not
known while you are writing the code. A class when the keys are **your design** and
you would write them out. `@dataclass` in module 12 makes the second one as short as
the first, and `exercises/thinking.md` asks you to draw the line before then.

## 7. The reference rule, again

A dict is mutable, so everything module 05 said about `b = a` holds unchanged.

In [ ]:
config = {"unit": "C"}
alias = config
alias["unit"] = "F"
print(config)  # one dict, two names

base = {"unit": "C"}
override = {"unit": "F", "scale": 1.8}

print(base | override)  # | builds a NEW dict, right-hand side wins on a clash
print(base)  # untouched

seen = base
base |= override  # |= changes the dict in place
print(seen)  # so the alias sees it

`|` and `|=` on dicts are the `+` and `+=` of module 05, one level up. `dict.copy()`
and `copy.deepcopy` behave exactly as they did for lists: the first copies one level,
the second copies all of them.

## 8. Sets

A set is a dict that kept only the keys: unique, hashable, unordered. Membership is
what it is for — `in` on a set does not depend on how many entries there are, while
`in` on a list walks it.

In [ ]:
faults = {"TH-04", "TH-09", "TH-04"}  # the duplicate does not survive
print(faults == {"TH-09", "TH-04"})  # comparing sets ignores order entirely
print(len(faults))

print(set())  # the empty set has no literal: {} is an empty DICT
print(type({}))
print(set([1, 2, 2, 3]))  # set() over a sequence is the way to remove duplicates

There is no order to rely on, and Python makes that visible: the hash of a string is
randomised per process, so a set of strings usually prints in a different arrangement
on the next run. Run the cell, then run it again after restarting the kernel. If the
arrangement does not move, check `PYTHONHASHSEED` — pinning it to a value switches
the randomisation off, which is what a build does when it wants reproducible output.

In [ ]:
print({"banana", "apple", "cherry", "date"})
print({3, 1, 2})  # small ints hash to themselves, so THIS one looks sorted -- it is not

The second line is the trap: `{3, 1, 2}` prints as `{1, 2, 3}` however often you run
it, because a small integer hashes to itself. That is an artefact of the
implementation, not a promise, and it is the reason the misconception survives —
people try it with numbers. When you need an order, ask for one with `sorted()`.

In [ ]:
today = {"TH-01", "TH-04", "TH-09"}
yesterday = {"TH-04", "TH-09", "TH-11"}

print(sorted(today & yesterday))  # in both
print(sorted(today | yesterday))  # in either
print(sorted(today - yesterday))  # new today
print(sorted(today ^ yesterday))  # in one but not the other
print(today <= yesterday)  # subset?

The same four operators work on `.keys()`, because a keys view is set-like. That is
the shortest way to ask what two dicts have in common.

In [ ]:
before = {"TH-01": 21.7, "TH-04": 91.0}
after = {"TH-04": 88.0, "TH-09": 23.1}

print(sorted(before.keys() & after.keys()))  # measured both times
print(sorted(after.keys() - before.keys()))  # appeared

---

`exercises/` is next: `exercise_01.py` to `exercise_06.py`, `exercise_09.py`, and two
in `thinking.md` with nothing to run.

One habit to take from section 8 into the exercises: never print a set directly when
something is going to compare the output. `sorted(...)` first.